In [21]:

from pathlib import Path
import zipfile
import shutil
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

# ============================================================
# Define project directories
# ============================================================

DATA_DIR = Path("/content/drive/MyDrive/movierec/data")
RAW_DIR = DATA_DIR / "raw"
MOVIELENS_DIR = RAW_DIR / "ml-32m"
ZIP_PATH = RAW_DIR / "ml-32m.zip"

RAW_DIR.mkdir(parents=True, exist_ok=True)
MOVIELENS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"MovieLens directory: {MOVIELENS_DIR}")

# ============================================================
# Verify the MovieLens archive
# ============================================================

if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"MovieLens archive not found: {ZIP_PATH}\n"
        "Upload ml-32m.zip to the raw directory on Google Drive."
    )

if not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError(f"Invalid ZIP archive: {ZIP_PATH}")

print("MovieLens archive found.")

# ============================================================
#  Extract the required CSV files
# ============================================================

required_files = ["links.csv", "movies.csv"]

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for filename in required_files:
        destination = MOVIELENS_DIR / filename
        archive_member = f"ml-32m/{filename}"

        if destination.is_file():
            print(f"Already extracted: {filename}")
            continue

        if archive_member not in archive.namelist():
            raise FileNotFoundError(
                f"Missing archive member: {archive_member}"
            )

        temporary_file = destination.with_suffix(".csv.part")

        try:
            with archive.open(archive_member) as source:
                with open(temporary_file, "wb") as target:
                    shutil.copyfileobj(source, target)

            temporary_file.replace(destination)
            print(f"Extracted: {filename}")

        finally:
            temporary_file.unlink(missing_ok=True)

# ============================================================
#  Load the MovieLens data
# ============================================================

links = pd.read_csv(
    MOVIELENS_DIR / "links.csv",
    dtype={
        "movieId": "int64",
        "imdbId": "string",
        "tmdbId": "string",
    }
)

movies = pd.read_csv(
    MOVIELENS_DIR / "movies.csv"
)

# ============================================================
# Validate and inspect the data
# ============================================================

assert links["movieId"].is_unique, "Duplicate movie IDs in links.csv"
assert movies["movieId"].is_unique, "Duplicate movie IDs in movies.csv"

print("\nMovieLens dataset loaded successfully.")
print(f"Total movies: {len(movies):,}")
print(f"Total movie links: {len(links):,}")
print(f"Movies with IMDb IDs: {links['imdbId'].notna().sum():,}")

print("\nMovies preview:")
display(movies.head())

print("\nLinks preview:")
display(links.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data directory: /content/drive/MyDrive/movierec/data
MovieLens directory: /content/drive/MyDrive/movierec/data/raw/ml-32m
MovieLens archive found.
Already extracted: links.csv
Already extracted: movies.csv

MovieLens dataset loaded successfully.
Total movies: 87,585
Total movie links: 87,585
Movies with IMDb IDs: 87,585

Movies preview:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Links preview:


,movieId,imdbId,tmdbId
0,1,0114709,862
1,2,0113497,8844
2,3,0113228,15602
3,4,0114885,31357
4,5,0113041,11862


In [22]:

# ============================================================
# Prepare a reproducible sample of 100 movies
# ============================================================

SAMPLE_SIZE = 100
RANDOM_STATE = 42

# Merge MovieLens metadata with IMDb identifiers
catalog = movies.merge(
    links,
    on="movieId",
    how="inner",
    validate="one_to_one"
)

# Exclude movies without IMDb identifiers
catalog = catalog.dropna(subset=["imdbId"]).copy()

# Add the IMDb title prefix
catalog["imdbId"] = (
    "tt" + catalog["imdbId"].str.zfill(7)
)

# Select a reproducible random sample
sample = catalog.sample(
    n=SAMPLE_SIZE,
    random_state=RANDOM_STATE
).copy()

sample = sample.sort_values("movieId").reset_index(drop=True)

print(f"Total movies in catalog: {len(catalog):,}")
print(f"Sample size: {len(sample)}")

display(sample[["movieId", "title", "imdbId"]].head(10))


# ============================================================
# Match IMDb IDs to Wikidata and English Wikipedia
# ============================================================

import time
import requests

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
BATCH_SIZE = 20

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "MovieRecResearch/0.1 "
        "(https://github.com/Cxsee/AI-in-Industry-project)"
    ),
    "Accept": "application/sparql-results+json"
})

def query_wikidata(imdb_ids):
    values = " ".join(f'"{imdb_id}"' for imdb_id in imdb_ids)

    query = f"""
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    PREFIX schema: <http://schema.org/>

    SELECT ?imdb ?item ?article WHERE {{
        VALUES ?imdb {{ {values} }}

        ?item wdt:P345 ?imdb .

        OPTIONAL {{
            ?article schema:about ?item ;
                     schema:isPartOf <https://en.wikipedia.org/> .
        }}
    }}
    """

    response = session.get(
        WIKIDATA_ENDPOINT,
        params={"query": query, "format": "json"},
        timeout=60
    )
    response.raise_for_status()

    return response.json()["results"]["bindings"]


records = []
imdb_ids = sample["imdbId"].tolist()

for start in range(0, len(imdb_ids), BATCH_SIZE):
    batch = imdb_ids[start:start + BATCH_SIZE]

    results = query_wikidata(batch)

    for result in results:
        records.append({
            "imdbId": result["imdb"]["value"],
            "wikidata_id": result["item"]["value"].split("/")[-1],
            "wikipedia_url": result.get(
                "article", {}
            ).get("value")
        })

    print(
        f"Processed {min(start + BATCH_SIZE, len(imdb_ids))}"
        f"/{len(imdb_ids)} IMDb IDs"
    )

    time.sleep(2)

# Build the mapping table
mapping = pd.DataFrame(
    records,
    columns=["imdbId", "wikidata_id", "wikipedia_url"]
)

pilot = sample.merge(mapping, on="imdbId", how="left")

# Check whether an IMDb ID matched multiple Wikidata items
ambiguous = mapping[
    mapping.duplicated("imdbId", keep=False)
]

wikidata_matches = pilot.loc[
    pilot["wikidata_id"].notna(), "movieId"
].nunique()

wikipedia_matches = pilot.loc[
    pilot["wikipedia_url"].notna(), "movieId"
].nunique()

print("\n--- Matching results ---")
print(f"Sampled movies: {len(sample)}")
print(f"Matched on Wikidata: {wikidata_matches}")
print(f"With English Wikipedia pages: {wikipedia_matches}")
print(f"Ambiguous IMDb IDs: {ambiguous['imdbId'].nunique()}")

display(
    pilot[
        ["movieId", "title", "wikidata_id", "wikipedia_url"]
    ].head(15)
)

# Save intermediate results
PILOT_DIR = DATA_DIR / "plots_pilot"
PILOT_DIR.mkdir(parents=True, exist_ok=True)

pilot.to_csv(
    PILOT_DIR / "wikidata_matches_100.csv",
    index=False
)

print("\nPilot mapping saved successfully.")



Total movies in catalog: 87,585
Sample size: 100


,movieId,title,imdbId
0,378,Speechless (1994),tt0111256
1,1194,Cheech and Chong's Up in Smoke (1978),tt0078446
2,2055,Hot Lead and Cold Feet (1978),tt0077698
3,4309,Petits Freres (Petits Frères) (1999),tt0184819
4,6384,Love the Hard Way (2001),tt0263671
5,7491,"Naked Prey, The (1966)",tt0060736
6,8545,Plain Dirty (a.k.a. Briar Patch) (2003),tt0220343
7,27378,Long Time Dead (2002),tt0251806
8,27685,Bring It On Again (2004),tt0334965
9,44255,Story of a Love Affair (Cronaca di un amore) (...,tt0042355


Processed 20/100 IMDb IDs
Processed 40/100 IMDb IDs
Processed 60/100 IMDb IDs
Processed 80/100 IMDb IDs
Processed 100/100 IMDb IDs

--- Matching results ---
Sampled movies: 100
Matched on Wikidata: 90
With English Wikipedia pages: 69
Ambiguous IMDb IDs: 0


,movieId,title,wikidata_id,wikipedia_url
0,378,Speechless (1994),Q1322346,https://en.wikipedia.org/wiki/Speechless_(1994...
1,1194,Cheech and Chong's Up in Smoke (1978),Q1381289,https://en.wikipedia.org/wiki/Up_in_Smoke
2,2055,Hot Lead and Cold Feet (1978),Q5910108,https://en.wikipedia.org/wiki/Hot_Lead_and_Col...
3,4309,Petits Freres (Petits Frères) (1999),Q16028797,None
4,6384,Love the Hard Way (2001),Q569321,https://en.wikipedia.org/wiki/Love_the_Hard_Way
5,7491,"Naked Prey, The (1966)",Q1197542,https://en.wikipedia.org/wiki/The_Naked_Prey
6,8545,Plain Dirty (a.k.a. Briar Patch) (2003),Q3644536,https://en.wikipedia.org/wiki/Briar_Patch_(film)
7,27378,Long Time Dead (2002),Q1517263,https://en.wikipedia.org/wiki/Long_Time_Dead
8,27685,Bring It On Again (2004),Q535505,https://en.wikipedia.org/wiki/Bring_It_On_Again
9,44255,Story of a Love Affair (Cronaca di un amore) (...,Q2222439,https://en.wikipedia.org/wiki/Story_of_a_Love_...



Pilot mapping saved successfully.


In [23]:

# ============================================================
# Extract plot sections from a pilot of 10 movies
# ============================================================

import re
import time
import requests

from urllib.parse import urlparse, unquote
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

PILOT_DIR = Path("/content/drive/MyDrive/movierec/data/plots_pilot")
WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"

# Load the mapping generated in the previous step
pilot = pd.read_csv(
    PILOT_DIR / "wikidata_matches_100.csv",
    dtype={"imdbId": "string", "wikidata_id": "string"}
)

# Select the first 10 movies with English Wikipedia pages
test_movies = pilot.dropna(
    subset=["wikipedia_url"]
).head(10).copy()

# Configure a reusable API session
wiki_session = requests.Session()
wiki_session.headers.update({
    "User-Agent": (
        "MovieRecResearch/0.1 "
        "(https://github.com/Cxsee/AI-in-Industry-project)"
    )
})

retry_policy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True
)

wiki_session.mount(
    "https://",
    HTTPAdapter(max_retries=retry_policy)
)

# Accepted Wikipedia section headings
PLOT_HEADINGS = {
    "plot",
    "plot summary",
    "synopsis",
    "premise",
    "story",
    "storyline",
    "story summary"
}


def wikipedia_request(params):
    """Send a request to the MediaWiki API."""
    response = wiki_session.get(
        WIKIPEDIA_API,
        params={
            **params,
            "format": "json",
            "maxlag": 5
        },
        timeout=(15, 45)
    )

    response.raise_for_status()
    data = response.json()

    if "error" in data:
        raise RuntimeError(str(data["error"]))

    return data["parse"]


def extract_plot(wikipedia_url):
    """Extract the plot section from a Wikipedia article."""

    # Convert the URL into a Wikipedia page title
    page_title = unquote(
        urlparse(wikipedia_url).path.split("/wiki/")[-1]
    ).replace("_", " ")

    # Retrieve the article sections
    parsed = wikipedia_request({
        "action": "parse",
        "page": page_title,
        "prop": "sections"
    })

    selected_section = None

    for section in parsed.get("sections", []):
        heading = BeautifulSoup(
            section["line"], "html.parser"
        ).get_text(" ", strip=True).lower()

        if heading in PLOT_HEADINGS:
            selected_section = section
            break

    if selected_section is None:
        return None, None, "no_plot_section"

    # Retrieve only the selected section
    parsed_text = wikipedia_request({
        "action": "parse",
        "page": page_title,
        "prop": "text",
        "section": selected_section["index"]
    })

    html = parsed_text["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")

    # Remove references, tables and non-narrative elements
    for element in soup.select(
        "sup.reference, table, .mw-editsection, "
        ".thumb, .hatnote, .navbox, style, script"
    ):
        element.decompose()

    # Extract paragraph text
    paragraphs = [
        p.get_text(" ", strip=True)
        for p in soup.find_all("p")
    ]

    plot = " ".join(p for p in paragraphs if p)
    plot = re.sub(r"\s+", " ", plot).strip()

    if len(plot) < 150:
        return plot, selected_section["line"], "short_plot"

    return plot, selected_section["line"], "extracted"


# Run the extraction pilot
results = []

for _, movie in test_movies.iterrows():
    try:
        plot, section, status = extract_plot(
            movie["wikipedia_url"]
        )

    except requests.RequestException as error:
        plot, section, status = None, None, "request_error"
        print(f"Request failed: {movie['title']} - {error}")

    except (RuntimeError, KeyError, ValueError) as error:
        plot, section, status = None, None, "api_error"
        print(f"API error: {movie['title']} - {error}")

    results.append({
        "movieId": movie["movieId"],
        "title": movie["title"],
        "imdbId": movie["imdbId"],
        "wikipedia_url": movie["wikipedia_url"],
        "section": section,
        "plot": plot,
        "plot_chars": len(plot) if plot else 0,
        "status": status
    })

    print(f"{movie['title']}: {status}")
    time.sleep(1)

plot_test = pd.DataFrame(results)

# Save intermediate results
plot_test.to_csv(
    PILOT_DIR / "plot_test_10.csv",
    index=False
)

print("\n--- Extraction results ---")
print(plot_test["status"].value_counts().to_string())
print("\nPilot results saved successfully.")


Speechless (1994): extracted
Cheech and Chong's Up in Smoke (1978): extracted
Hot Lead and Cold Feet (1978): extracted
Love the Hard Way (2001): no_plot_section
Naked Prey, The (1966): extracted
Plain Dirty (a.k.a. Briar Patch) (2003): extracted
Long Time Dead (2002): extracted
Bring It On Again (2004): extracted
Story of a Love Affair (Cronaca di un amore) (1950): extracted
Street Kings (2008): extracted

--- Extraction results ---
status
extracted          9
no_plot_section    1

Pilot results saved successfully.


In [24]:

# ============================================================
# Inspect extracted plots
# ============================================================

preview = plot_test[
    ["movieId", "title", "section", "plot_chars", "status"]
].copy()

display(preview)

# Show the first three successfully extracted plots
successful = plot_test[
    plot_test["status"] == "extracted"
].head(3)

for _, movie in successful.iterrows():
    print("=" * 70)
    print(f"Movie: {movie['title']}")
    print(f"Section: {movie['section']}")
    print(f"Source: {movie['wikipedia_url']}")
    print(f"\nPlot preview:\n{movie['plot'][:700]}")
    print()


,movieId,title,section,plot_chars,status
0,378,Speechless (1994),Plot,2690,extracted
1,1194,Cheech and Chong's Up in Smoke (1978),Plot,3607,extracted
2,2055,Hot Lead and Cold Feet (1978),Plot,5000,extracted
3,6384,Love the Hard Way (2001),None,0,no_plot_section
4,7491,"Naked Prey, The (1966)",Plot summary,3246,extracted
5,8545,Plain Dirty (a.k.a. Briar Patch) (2003),Plot,6723,extracted
6,27378,Long Time Dead (2002),Plot,3668,extracted
7,27685,Bring It On Again (2004),Plot,1810,extracted
8,44255,Story of a Love Affair (Cronaca di un amore) (...,Plot,4207,extracted
9,59016,Street Kings (2008),Plot,3157,extracted


Movie: Speechless (1994)
Section: Plot
Source: https://en.wikipedia.org/wiki/Speechless_(1994_film)

Plot preview:
After meeting each other for the first time in the gift shop of the Las Cruces, New Mexico hotel where they are both staying, insomniac writers Julia Mann and Kevin Vallick fight over the last box of sleep medicine. Initially, they do not realize both are speechwriters for rival candidates in a New Mexico Senate election, so they both fall in love. Julia works for Democratic candidate Lloyd Wannamaker and Kevin for Republican candidate Ray Garvin. Inadvertently influencing each other's work, they discover each other's jobs when they are both invited to speak at a school. Both shocked, they end up attacking each other verbally on stage. Later, in their campaign speeches, they continue to argu

Movie: Cheech and Chong's Up in Smoke (1978)
Section: Plot
Source: https://en.wikipedia.org/wiki/Up_in_Smoke

Plot preview:
Anthony "Man" Stoner, an unemployed, marijuana-smoking drum

In [25]:

# ============================================================
# Complete plot extraction for the 100-movie pilot
# ============================================================

import time
import requests

# Load the previously generated Wikidata mappings
pilot = pd.read_csv(
    PILOT_DIR / "wikidata_matches_100.csv",
    dtype={"imdbId": "string", "wikidata_id": "string"}
)

RESULTS_PATH = PILOT_DIR / "plot_extraction_100.csv"

# Resume previous extraction results if available
if RESULTS_PATH.exists():
    extraction = pd.read_csv(RESULTS_PATH)
else:
    extraction = pd.read_csv(PILOT_DIR / "plot_test_10.csv")

# Process only movies with English Wikipedia pages
candidates = pilot.dropna(subset=["wikipedia_url"]).copy()

# Retry previously failed API requests, if any
completed_ids = set(
    extraction.loc[
        ~extraction["status"].isin(["request_error", "api_error"]),
        "movieId"
    ]
)

remaining = candidates[
    ~candidates["movieId"].isin(completed_ids)
]

print(f"Movies with Wikipedia pages: {len(candidates)}")
print(f"Already processed: {len(candidates) - len(remaining)}")
print(f"Remaining movies: {len(remaining)}")

new_records = []

for index, (_, movie) in enumerate(remaining.iterrows(), start=1):

    try:
        plot, section, status = extract_plot(
            movie["wikipedia_url"]
        )

    except requests.RequestException as error:
        plot, section, status = None, None, "request_error"
        print(f"Request error: {movie['title']} - {error}")

    except (RuntimeError, KeyError, ValueError) as error:
        plot, section, status = None, None, "api_error"
        print(f"API error: {movie['title']} - {error}")

    new_records.append({
        "movieId": movie["movieId"],
        "title": movie["title"],
        "imdbId": movie["imdbId"],
        "wikipedia_url": movie["wikipedia_url"],
        "section": section,
        "plot": plot,
        "plot_chars": len(plot) if plot else 0,
        "status": status
    })

    print(
        f"[{index}/{len(remaining)}] "
        f"{movie['title']}: {status}"
    )

    # Save progress periodically
    if index % 10 == 0:
        extraction = pd.concat(
            [extraction, pd.DataFrame(new_records)],
            ignore_index=True
        )
        extraction = extraction.drop_duplicates(
            subset=["movieId"],
            keep="last"
        )
        extraction.to_csv(RESULTS_PATH, index=False)
        new_records = []

    time.sleep(1)

# Save remaining results
if new_records:
    extraction = pd.concat(
        [extraction, pd.DataFrame(new_records)],
        ignore_index=True
    )
    extraction = extraction.drop_duplicates(
        subset=["movieId"],
        keep="last"
    )

extraction.to_csv(RESULTS_PATH, index=False)

# Merge results with the full 100-movie sample
plot_100 = pilot.merge(
    extraction[
        ["movieId", "section", "plot", "plot_chars", "status"]
    ],
    on="movieId",
    how="left",
    validate="one_to_one"
)

# Assign statuses to movies without Wikipedia pages
plot_100.loc[
    plot_100["wikidata_id"].isna(), "status"
] = "no_wikidata_match"

plot_100.loc[
    plot_100["wikidata_id"].notna()
    & plot_100["wikipedia_url"].isna(),
    "status"
] = "no_en_wikipedia"

plot_100["status"] = plot_100["status"].fillna("pending")

# Save the complete pilot dataset
plot_100.to_csv(
    PILOT_DIR / "plots_pilot_100.csv",
    index=False
)

print("\n--- Final pilot results ---")
print(plot_100["status"].value_counts().to_string())

extracted_count = (plot_100["status"] == "extracted").sum()

print(
    f"\nPlots extracted: "
    f"{extracted_count}/{len(plot_100)}"
)

print(
    f"Overall coverage: "
    f"{extracted_count / len(plot_100):.1%}"
)

print("\nPilot dataset saved successfully.")


Movies with Wikipedia pages: 69
Already processed: 69
Remaining movies: 0

--- Final pilot results ---
status
extracted            56
no_en_wikipedia      21
no_wikidata_match    10
no_plot_section       9
short_plot            4

Plots extracted: 56/100
Overall coverage: 56.0%

Pilot dataset saved successfully.


In [26]:

# ============================================================
# Inspect plot extraction quality
# ============================================================

PILOT_DIR = Path(
    "/content/drive/MyDrive/movierec/data/plots_pilot"
)

plot_100 = pd.read_csv(
    PILOT_DIR / "plots_pilot_100.csv"
)

# Select random successful extractions
successful = plot_100[
    plot_100["status"] == "extracted"
]

success_sample = successful.sample(
    n=min(8, len(successful)),
    random_state=42
)

# Include all short plots
short_plots = plot_100[
    plot_100["status"] == "short_plot"
]

# Include some movies without a recognized plot section
missing_sections = plot_100[
    plot_100["status"] == "no_plot_section"
].head(3)

# Combine all review candidates
review = pd.concat(
    [success_sample, short_plots, missing_sections],
    ignore_index=True
).drop_duplicates(subset="movieId")

# Create text previews
review["plot_preview"] = (
    review["plot"]
    .fillna("")
    .str.slice(0, 800)
)

review["quality_label"] = ""

# Display the review table
display(
    review[
        [
            "movieId",
            "title",
            "status",
            "plot_chars",
            "plot_preview",
            "wikipedia_url"
        ]
    ]
)

# Save candidates for manual review
review.to_csv(
    PILOT_DIR / "manual_quality_review.csv",
    index=False
)

print(f"Movies selected for review: {len(review)}")
print(f"Successful extractions sampled: {len(success_sample)}")
print(f"Short plots included: {len(short_plots)}")
print(f"Missing plot sections sampled: {len(missing_sections)}")
print("Manual review file saved successfully.")


,movieId,title,status,plot_chars,plot_preview,wikipedia_url
0,378,Speechless (1994),extracted,2690.0,After meeting each other for the first time in...,https://en.wikipedia.org/wiki/Speechless_(1994...
1,27378,Long Time Dead (2002),extracted,3668.0,"In 1979, a group of demon worshippers in Moroc...",https://en.wikipedia.org/wiki/Long_Time_Dead
2,186277,Solsidan (2017),extracted,6085.0,"On Christmas Eve , dentist Alexander ""Alex"" Lö...",https://en.wikipedia.org/wiki/Solsidan_(film)
3,92029,Westbound (1959),extracted,1752.0,"In 1864, Union army officer Captain John Hayes...",https://en.wikipedia.org/wiki/Westbound_(film)
4,130942,Cool it Carol (1972),extracted,489.0,Young couple Joe and Carol leave the sticks be...,https://en.wikipedia.org/wiki/Cool_It_Carol!
5,261975,Mickey's Rival (1936),extracted,3075.0,"Mickey Mouse and his girlfriend, Minnie , are ...",https://en.wikipedia.org/wiki/Mickey%27s_Rival
6,199035,Blood Money (2017),extracted,1169.0,"During a weekend excursion in the woods, three...",https://en.wikipedia.org/wiki/Blood_Money_(201...
7,158986,Criminal Court (1946),extracted,796.0,Hotshot lawyer Steve Barnes is a candidate to ...,https://en.wikipedia.org/wiki/Criminal_Court_(...
8,160748,Cowboy Canteen (1944),short_plot,50.0,Entertainers perform on a dude ranch for soldi...,https://en.wikipedia.org/wiki/Cowboy_Canteen
9,171513,Crash! (1976),short_plot,125.0,Jealous invalid husband (Ferrer) tries to kill...,https://en.wikipedia.org/wiki/Crash!


Movies selected for review: 15
Successful extractions sampled: 8
Short plots included: 4
Missing plot sections sampled: 3
Manual review file saved successfully.


In [27]:

# ============================================================
# Diagnose missing and short plot extractions
# ============================================================


PILOT_DIR = Path(
    "/content/drive/MyDrive/movierec/data/plots_pilot"
)

plot_100 = pd.read_csv(
    PILOT_DIR / "plots_pilot_100.csv"
)

# Select movies with incomplete extraction
failed = plot_100[
    plot_100["status"].isin([
        "no_plot_section",
        "short_plot"
    ])
].copy()


def get_page_title(url):
    """Convert a Wikipedia URL into a page title."""
    path = urlparse(url).path
    return unquote(path.split("/wiki/")[-1]).replace("_", " ")


def clean_heading(html):
    """Remove HTML markup from a section heading."""
    return BeautifulSoup(
        html, "html.parser"
    ).get_text(" ", strip=True)


for _, movie in failed.iterrows():

    print("=" * 65)
    print(f"Movie: {movie['title']}")
    print(f"Status: {movie['status']}")

    try:
        page_title = get_page_title(
            movie["wikipedia_url"]
        )

        parsed = wikipedia_request({
            "action": "parse",
            "page": page_title,
            "prop": "sections"
        })

        sections = parsed.get("sections", [])

        headings = [
            clean_heading(section["line"])
            for section in sections
        ]

        print("Available sections:")
        print(headings if headings else "No sections found")

        if movie["status"] == "short_plot":
            print("\nExtracted text:")
            print(
                movie["plot"]
                if pd.notna(movie["plot"])
                else "(empty)"
            )

            # Inspect HTML when a recognized plot is empty
            if pd.isna(movie["plot"]) or movie["plot_chars"] == 0:
                selected = next(
                    (
                        s for s in sections
                        if clean_heading(s["line"]).lower()
                        in PLOT_HEADINGS
                    ),
                    None
                )

                if selected is not None:
                    section_data = wikipedia_request({
                        "action": "parse",
                        "page": page_title,
                        "prop": "text",
                        "section": selected["index"]
                    })

                    html = section_data["text"]["*"]
                    soup = BeautifulSoup(html, "html.parser")

                    print("\nRaw section text preview:")
                    print(soup.get_text(" ", strip=True)[:500])
                    print(
                        "HTML paragraphs:",
                        len(soup.find_all("p"))
                    )

    except Exception as error:
        print(f"Inspection failed: {error}")

    time.sleep(1)

print("\nDiagnostic inspection completed.")


Movie: Love the Hard Way (2001)
Status: no_plot_section
Available sections:
['Cast', 'Reception', 'References', 'External links']
Movie: Manslaughter (Drabet) (2005)
Status: no_plot_section
Available sections:
['Cast', 'References', 'External links']
Movie: Service (Serbis) (2008)
Status: no_plot_section
Available sections:
['Cast', 'Production', 'Release', 'Reception', 'Critical response', 'Accolades', 'References', 'External links']
Movie: Nicky's Family (2011)
Status: no_plot_section
Available sections:
['Cast', 'References', 'External links']
Movie: The Skulls III (2004)
Status: no_plot_section
Available sections:
['Cast', 'Reception', 'Accolades', 'References', 'External links']
Movie: Cowboy Canteen (1944)
Status: short_plot
Available sections:
['Plot', 'Cast', 'See also', 'References', 'External links']

Extracted text:
Entertainers perform on a dude ranch for soldiers.
Movie: Wolf Creek (2016)
Status: no_plot_section
Available sections:
['Plot overview', 'Season 1', 'Season 2',

In [28]:

# ============================================================
# Inspect article introductions for missing plots
# ============================================================

import time
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse, unquote

# Load the complete pilot results
plot_100 = pd.read_csv(
    PILOT_DIR / "plots_pilot_100.csv"
)

# Select movies without a usable plot
candidates = plot_100[
    plot_100["status"].isin([
        "no_plot_section",
        "short_plot"
    ])
].copy()


def extract_lead_text(wikipedia_url):
    """Extract introductory paragraphs from a Wikipedia article."""

    page_title = unquote(
        urlparse(wikipedia_url).path.split("/wiki/")[-1]
    ).replace("_", " ")

    # Section 0 contains the article introduction
    parsed = wikipedia_request({
        "action": "parse",
        "page": page_title,
        "prop": "text",
        "section": 0
    })

    html = parsed["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")

    # Remove non-narrative elements
    for element in soup.select(
        "table, .infobox, .hatnote, .ambox, "
        "sup.reference, .mw-editsection, "
        "figure, style, script"
    ):
        element.decompose()

    paragraphs = [
        p.get_text(" ", strip=True)
        for p in soup.find_all("p")
    ]

    paragraphs = [p for p in paragraphs if p]

    return " ".join(paragraphs).strip()


lead_results = []

for _, movie in candidates.iterrows():

    try:
        lead = extract_lead_text(
            movie["wikipedia_url"]
        )

        status = "lead_available" if lead else "empty_lead"

    except Exception as error:
        lead = ""
        status = "request_error"
        print(f"Error: {movie['title']} - {error}")

    lead_results.append({
        "movieId": movie["movieId"],
        "title": movie["title"],
        "original_status": movie["status"],
        "lead_text": lead,
        "lead_chars": len(lead),
        "lead_preview": lead[:600],
        "lead_status": status
    })

    time.sleep(1)

lead_review = pd.DataFrame(lead_results)

display(
    lead_review[
        [
            "movieId",
            "title",
            "original_status",
            "lead_chars",
            "lead_preview"
        ]
    ]
)

# Save diagnostic results
lead_review.to_csv(
    PILOT_DIR / "lead_review_13.csv",
    index=False
)

print("\n--- Introduction analysis ---")
print(lead_review["lead_status"].value_counts())
print("\nDiagnostic results saved successfully.")


,movieId,title,original_status,lead_chars,lead_preview
0,6384,Love the Hard Way (2001),no_plot_section,443,Love the Hard Way is a 2001 crime drama film d...
1,60544,Manslaughter (Drabet) (2005),no_plot_section,711,Manslaughter ( Danish : Drabet ) is a 2005 Dan...
2,86500,Service (Serbis) (2008),no_plot_section,436,Service ( Filipino : Serbis ) is a 2008 Filipi...
3,114854,Nicky's Family (2011),no_plot_section,176,Nicky's Family ( Czech : Nickyho rodina ) is a...
4,139801,The Skulls III (2004),no_plot_section,343,"The Skulls III is a 2004 thriller film , direc..."
5,160748,Cowboy Canteen (1944),short_plot,110,Cowboy Canteen is a 1944 American musical West...
6,163921,Wolf Creek (2016),no_plot_section,505,Wolf Creek is an Australian horror television ...
7,171513,Crash! (1976),short_plot,151,Crash! is a 1977 film directed by Charles Band...
8,177961,Big Shots (1982),no_plot_section,853,Mitahat La'af (also known as Under The Nose ) ...
9,188591,Death and the Compass (1992),short_plot,577,Death and the Compass is a 1996 American thril...



--- Introduction analysis ---
lead_status
lead_available    13
Name: count, dtype: int64

Diagnostic results saved successfully.


In [29]:

# ============================================================
# Review introduction content for missing plot sections
# ============================================================

lead_review = pd.read_csv(
    PILOT_DIR / "lead_review_13.csv"
)

# Focus on movies without a recognized plot section
candidates = lead_review[
    lead_review["original_status"] == "no_plot_section"
].copy()

print(f"Movies requiring introduction review: {len(candidates)}")

for _, movie in candidates.iterrows():

    print("\n" + "=" * 80)
    print(f"Movie: {movie['title']}")
    print(f"MovieLens ID: {movie['movieId']}")
    print(f"Introduction length: {movie['lead_chars']} characters")

    print("\nFULL INTRODUCTION:\n")
    print(movie["lead_text"])

    print("\n" + "-" * 80)

# Prepare a manual evaluation table
quality_review = candidates[
    ["movieId", "title", "lead_chars", "lead_text"]
].copy()

quality_review["identity_match"] = ""
quality_review["contains_plot"] = ""
quality_review["notes"] = ""

quality_review.to_csv(
    PILOT_DIR / "introduction_quality_review.csv",
    index=False
)

print("\nManual review template saved successfully.")


Movies requiring introduction review: 9

Movie: Love the Hard Way (2001)
MovieLens ID: 6384
Introduction length: 443 characters

FULL INTRODUCTION:

Love the Hard Way is a 2001 crime drama film directed by Peter Sehr [ fr ] . The story revolves around a petty thief who brings an innocent young woman into his world of crime while she teaches him the lessons of enjoying life and being loved. The film was written by Peter Sehr [ fr ] and is based on the novel Yi Ban Shi Huo Yan, Yi Ban Shi Hai Shui by Chinese writer Wang Shuo , although Sehr transported the film's story to New York City .

--------------------------------------------------------------------------------

Movie: Manslaughter (Drabet) (2005)
MovieLens ID: 60544
Introduction length: 711 characters

FULL INTRODUCTION:

Manslaughter ( Danish : Drabet ) is a 2005 Danish drama film written and directed by Per Fly . The film stars Jesper Christensen , Beate Bille and Pernilla August as well as Fly's wife Charlotte Fich . Manslaugh

In [30]:

# ============================================================
# Test an extended set of plot headings
# ============================================================

# Include the additional heading identified during diagnosis
PLOT_HEADINGS.add("plot overview")

# Select Wolf Creek from the pilot dataset
wolf_creek = plot_100.loc[
    plot_100["movieId"] == 163921
].iloc[0]

# Extract the plot using the updated heading set
plot, section, status = extract_plot(
    wolf_creek["wikipedia_url"]
)

print(f"Movie: {wolf_creek['title']}")
print(f"Detected section: {section}")
print(f"Extraction status: {status}")
print(f"Plot length: {len(plot) if plot else 0} characters")

print("\nExtracted plot:\n")
print(plot if plot else "No plot extracted.")


Movie: Wolf Creek (2016)
Detected section: Plot overview
Extraction status: extracted
Plot length: 599 characters

Extracted plot:

The Thorogood family, who is vacationing in Australia, is attacked and annihilated by Mick Taylor, save only Eve, their 19-year-old daughter. Eve thus embarks on a mission of revenge through The Australian outback to find and kill Taylor, helped by Det. Sullivan Hill. An outback safari bus, bringing a group of international tourists, is abducted by Mick Taylor who drugs the group and leaves them in the middle of the Southern outback desert, disabling (and later blowing up) the bus. The survivors go on a desperate journey through the outback, fighting for their lives as Taylor hunts them down.


In [31]:

# ============================================================
# Update the pilot dataset with the recovered plot
# ============================================================

PLOT_HEADINGS.add("plot overview")

# Load the original pilot results
pilot_v2 = pd.read_csv(
    PILOT_DIR / "plots_pilot_100.csv"
)

# Recover the Wolf Creek plot
movie_id = 163921

movie = pilot_v2.loc[
    pilot_v2["movieId"] == movie_id
].iloc[0]

plot, section, status = extract_plot(
    movie["wikipedia_url"]
)

# Update the record only if extraction succeeds
if status == "extracted" and plot:

    mask = pilot_v2["movieId"] == movie_id

    pilot_v2.loc[mask, "plot"] = plot
    pilot_v2.loc[mask, "section"] = section
    pilot_v2.loc[mask, "plot_chars"] = len(plot)
    pilot_v2.loc[mask, "status"] = status

    # Preserve the source of the recovered text
    if "plot_source" not in pilot_v2.columns:
        pilot_v2["plot_source"] = pd.NA

    pilot_v2.loc[mask, "plot_source"] = (
        "wikipedia_plot_overview"
    )

    # Save a new version without replacing the original
    output_path = PILOT_DIR / "plots_pilot_100_v2.csv"
    pilot_v2.to_csv(output_path, index=False)

    print("Recovered plot saved successfully.")
    print(f"File: {output_path}")

else:
    print(f"Plot recovery unsuccessful: {status}")

# Display updated statistics
if status == "extracted" and plot:
    print("\n--- Updated pilot statistics ---")
    print(pilot_v2["status"].value_counts().to_string())

    coverage = (
        (pilot_v2["status"] == "extracted").sum()
        / len(pilot_v2)
    )

    print(f"\nOverall coverage: {coverage:.1%}")


Recovered plot saved successfully.
File: /content/drive/MyDrive/movierec/data/plots_pilot/plots_pilot_100_v2.csv

--- Updated pilot statistics ---
status
extracted            57
no_en_wikipedia      21
no_wikidata_match    10
no_plot_section       8
short_plot            4

Overall coverage: 57.0%


In [32]:

# ============================================================
# Consolidate validated plot recovery improvements
# ============================================================

import pandas as pd

# Load the latest pilot version
pilot_v3 = pd.read_csv(
    PILOT_DIR / "plots_pilot_100_v2.csv"
)

lead_review = pd.read_csv(
    PILOT_DIR / "lead_review_13.csv"
).set_index("movieId")

# Track the source of each text
if "plot_source" not in pilot_v3.columns:
    pilot_v3["plot_source"] = pd.NA

standard_mask = (
    (pilot_v3["status"] == "extracted")
    & pilot_v3["plot_source"].isna()
)

pilot_v3.loc[
    standard_mask, "plot_source"
] = "wikipedia_plot_section"

# Accept manually reviewed short plot descriptions
accepted_short_ids = [
    160748,  # Cowboy Canteen
    171513,  # Crash!
    214764   # Dance With Me
]

for movie_id in accepted_short_ids:
    mask = pilot_v3["movieId"] == movie_id

    if not mask.any():
        raise ValueError(f"Movie not found: {movie_id}")

    plot = pilot_v3.loc[mask, "plot"].iloc[0]

    if pd.isna(plot) or not str(plot).strip():
        raise ValueError(f"Empty short plot: {movie_id}")

    pilot_v3.loc[mask, "status"] = "short_plot_accepted"
    pilot_v3.loc[mask, "plot_source"] = "wikipedia_short_plot"


def extract_reviewed_passage(movie_id, start, end=None):
    """Select a reviewed narrative passage from an article lead."""

    text = lead_review.loc[movie_id, "lead_text"]

    start_pos = text.find(start)

    if start_pos == -1:
        raise ValueError(
            f"Expected narrative passage not found: {movie_id}"
        )

    end_pos = text.find(end, start_pos) if end else len(text)

    if end and end_pos == -1:
        raise ValueError(
            f"Expected passage boundary not found: {movie_id}"
        )

    return text[start_pos:end_pos].strip()


# Extract narrative sentences from manually reviewed leads
reviewed_synopses = {
    6384: extract_reviewed_passage(
        6384,
        "The story revolves around",
        "The film was written"
    ),
    177961: extract_reviewed_passage(
        177961,
        "The film is about"
    )
}

for movie_id, synopsis in reviewed_synopses.items():
    mask = pilot_v3["movieId"] == movie_id

    pilot_v3.loc[mask, "plot"] = synopsis
    pilot_v3.loc[mask, "plot_chars"] = len(synopsis)
    pilot_v3.loc[mask, "status"] = "lead_synopsis_reviewed"
    pilot_v3.loc[mask, "plot_source"] = "wikipedia_lead"

# Identify movies with usable candidate text
usable_statuses = {
    "extracted",
    "short_plot_accepted",
    "lead_synopsis_reviewed"
}

pilot_v3["has_candidate_text"] = (
    pilot_v3["status"].isin(usable_statuses)
    & pilot_v3["plot"].fillna("").str.strip().ne("")
)

# Save the consolidated pilot without overwriting earlier versions
output_path = PILOT_DIR / "plots_pilot_100_v3.csv"
pilot_v3.to_csv(output_path, index=False)

# Report results
print("--- Consolidated pilot results ---")
print(pilot_v3["status"].value_counts().to_string())

usable_count = int(pilot_v3["has_candidate_text"].sum())

print(f"\nMovies with candidate text: {usable_count}/100")
print(f"Candidate text coverage: {usable_count / len(pilot_v3):.1%}")
print(f"Saved to: {output_path}")

display(
    pilot_v3.loc[
        pilot_v3["has_candidate_text"],
        ["movieId", "title", "status", "plot_source", "plot_chars"]
    ].tail(10)
)


--- Consolidated pilot results ---
status
extracted                 57
no_en_wikipedia           21
no_wikidata_match         10
no_plot_section            6
short_plot_accepted        3
lead_synopsis_reviewed     2
short_plot                 1

Movies with candidate text: 62/100
Candidate text coverage: 62.0%
Saved to: /content/drive/MyDrive/movierec/data/plots_pilot/plots_pilot_100_v3.csv


,movieId,title,status,plot_source,plot_chars
87,242908,The Seventh Day (2021),extracted,wikipedia_plot_section,1129.0
88,243034,A Tale of Mari and Three Puppies (2007),extracted,wikipedia_plot_section,3787.0
90,249694,So Much for So Little (1949),extracted,wikipedia_plot_section,828.0
92,261503,"About Vitya, Masha, and Marines (1973)",extracted,wikipedia_plot_section,556.0
93,261975,Mickey's Rival (1936),extracted,wikipedia_plot_section,3075.0
95,270378,Zero Fucks Given (2022),extracted,wikipedia_plot_section,3557.0
96,273653,Agent Game (2022),extracted,wikipedia_plot_section,291.0
97,276099,Nikamma (2022),extracted,wikipedia_plot_section,4051.0
98,287239,I'd Love to Take Orders from You (1936),extracted,wikipedia_plot_section,1448.0
99,287685,The Beloved Rogue (1927),extracted,wikipedia_plot_section,2084.0


In [34]:

# ============================================================
# Find non-English Wikipedia pages for missing movies
# ============================================================

from pathlib import Path
from urllib.parse import urlparse
from collections import Counter
import time
import requests
import pandas as pd

PILOT_DIR = Path(
    "/content/drive/MyDrive/movierec/data/plots_pilot"
)

pilot = pd.read_csv(
    PILOT_DIR / "plots_pilot_100_v3.csv",
    dtype={"wikidata_id": "string"}
)

# Select movies without an English Wikipedia page
missing = pilot.loc[
    pilot["status"] == "no_en_wikipedia"
].copy()

print(f"Movies to investigate: {len(missing)}")

# Preferred languages for possible future extraction
LANGUAGE_PRIORITY = [
    "it", "fr", "es", "de", "pt",
    "nl", "ru", "pl", "ja", "zh"
]

WIKIDATA_API = "https://www.wikidata.org/w/api.php"

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "MovieRecResearch/0.1 "
        "(https://github.com/Cxsee/AI-in-Industry-project)"
    )
})

ids = missing["wikidata_id"].dropna().unique().tolist()
entities = {}

# Fetch sitelinks in small batches
for start in range(0, len(ids), 20):
    batch = ids[start:start + 20]

    response = session.get(
        WIKIDATA_API,
        params={
            "action": "wbgetentities",
            "ids": "|".join(batch),
            "props": "sitelinks/urls",
            "format": "json",
            "maxlag": 5
        },
        timeout=(15, 60)
    )
    response.raise_for_status()

    payload = response.json()

    if "error" in payload:
        raise RuntimeError(payload["error"])

    entities.update(payload["entities"])

    print(f"Retrieved {min(start + 20, len(ids))}/{len(ids)} entities")
    time.sleep(1)

# Build a table with all available Wikipedia languages
records = []

for _, movie in missing.iterrows():
    entity = entities.get(movie["wikidata_id"], {})

    if not entity or "missing" in entity:
        raise RuntimeError(
            f"Wikidata entity unavailable: {movie['wikidata_id']}"
        )

    for site, details in entity.get("sitelinks", {}).items():
        url = details.get("url", "")
        host = urlparse(url).hostname or ""

        # Keep Wikipedia articles, excluding other Wikimedia projects
        if not host.endswith(".wikipedia.org"):
            continue

        language = host.removesuffix(".wikipedia.org")

        if language == "en":
            continue

        records.append({
            "movieId": movie["movieId"],
            "title": movie["title"],
            "wikidata_id": movie["wikidata_id"],
            "language": language,
            "wikipedia_url": url
        })

language_links = pd.DataFrame(
    records,
    columns=[
        "movieId", "title", "wikidata_id",
        "language", "wikipedia_url"
    ]
)

# Summarize available languages for each movie
summary = []

for _, movie in missing.iterrows():
    available = language_links.loc[
        language_links["movieId"] == movie["movieId"]
    ].copy()

    available["priority"] = available["language"].map(
        lambda lang: (
            LANGUAGE_PRIORITY.index(lang)
            if lang in LANGUAGE_PRIORITY else 999
        )
    )

    available = available.sort_values(
        ["priority", "language"]
    )

    best = available.iloc[0] if not available.empty else None

    summary.append({
        "movieId": movie["movieId"],
        "title": movie["title"],
        "language_count": len(available),
        "preferred_language": (
            best["language"] if best is not None else None
        ),
        "preferred_url": (
            best["wikipedia_url"] if best is not None else None
        )
    })

language_summary = pd.DataFrame(summary)

# Save intermediate results
language_links.to_csv(
    PILOT_DIR / "multilingual_links_all.csv",
    index=False
)

language_summary.to_csv(
    PILOT_DIR / "multilingual_summary_21.csv",
    index=False
)

# Report results
found = (language_summary["language_count"] > 0).sum()

print("\n--- Multilingual Wikipedia coverage ---")
print(f"Movies investigated: {len(missing)}")
print(f"Movies with another Wikipedia language: {found}")
print(f"Movies without any Wikipedia page: {len(missing) - found}")

print("\nAvailable languages:")
print(language_links["language"].value_counts().head(15))

display(language_summary)

print("\nMultilingual analysis saved successfully.")


Movies to investigate: 21
Retrieved 20/21 entities
Retrieved 21/21 entities

--- Multilingual Wikipedia coverage ---
Movies investigated: 21
Movies with another Wikipedia language: 17
Movies without any Wikipedia page: 4

Available languages:
language
cy    12
fr     9
de     4
it     4
ru     2
ht     1
ur     1
ca     1
fi     1
ja     1
nl     1
an     1
es     1
sv     1
sr     1
Name: count, dtype: int64


,movieId,title,language_count,preferred_language,preferred_url
0,4309,Petits Freres (Petits Frères) (1999),2,fr,https://fr.wikipedia.org/wiki/Petits_Fr%C3%A8res
1,73744,If You Love (Jos rakastat) (2010),2,cy,https://cy.wikipedia.org/wiki/Jos_Rakastat
2,101188,Central Park (1990),0,None,None
3,107147,Last Winter (L'hiver dernier) (2011),3,fr,https://fr.wikipedia.org/wiki/L%27Hiver_dernier
4,126652,Raven the Little Rascal (2012),3,fr,https://fr.wikipedia.org/wiki/Petit_Corbeau_(f...
5,138364,Una donna per la vita (2012),2,it,https://it.wikipedia.org/wiki/Una_donna_per_la...
6,145288,A Deadly Encounter (2004),2,fr,https://fr.wikipedia.org/wiki/Pi%C3%A8ge_pour_...
7,151611,Addicted to Sexting (2015),1,cy,https://cy.wikipedia.org/wiki/Addicted_to_Sexting
8,152858,Flyin' Ryan (2003),0,None,None
9,154266,Critique de la séparation (1961),3,fr,https://fr.wikipedia.org/wiki/Critique_de_la_s...



Multilingual analysis saved successfully.


## Multilingual plot recovery

Continue here in Colab after mounting Google Drive. This section reads
`plots_pilot_100_v3.csv` and `multilingual_links_all.csv` from the same
`movierec/data/plots_pilot` directory. Run the sitelink discovery cell above if
the links CSV is missing or needs to be refreshed. If the pilot is missing, run
the earlier pilot cells first.

1. Inspect the first five preferred non-Welsh articles and review their headings.
2. For each film without an English Wikipedia page, try the available language
   editions in priority order. Select the first explicit plot or synopsis section
   with at least 150 characters of cleaned prose, using the pilot's existing
   threshold. Short extracts are saved in the audit for manual review.
3. Measure the observed candidate-text coverage and write
   `plots_pilot_100_v4.csv` and `multilingual_plot_audit.csv`.
   The audit distinguishes articles with no usable section from API failures
   and films with no available sitelink.

The coverage counts source-language candidates. New texts keep their original
language and revision URL so they can be reviewed before translation or use
with multilingual embeddings. Introductions are excluded from automatic
acceptance because they often contain production details.

In a manual check of the first five preferred articles from the saved
discovery results, the heading was **Synopsis** for
[Petits Frères](https://fr.wikipedia.org/wiki/Petits_Fr%C3%A8res),
[Last Winter](https://fr.wikipedia.org/wiki/L%27Hiver_dernier),
[Raven the Little Rascal](https://fr.wikipedia.org/wiki/Petit_Corbeau_(film))
and [A Deadly Encounter](https://fr.wikipedia.org/wiki/Pi%C3%A8ge_pour_une_m%C3%A8re),
and **Trama** for
[Una donna per la vita](https://it.wikipedia.org/wiki/Una_donna_per_la_vita).


In [ ]:
# 21. Inspect multilingual Wikipedia article structure
from pathlib import Path
import re
import time
import unicodedata
from urllib.parse import unquote, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

PILOT_DIR = Path("/content/drive/MyDrive/movierec/data/plots_pilot")
BASE_PATH = PILOT_DIR / "plots_pilot_100_v3.csv"
LINKS_PATH = PILOT_DIR / "multilingual_links_all.csv"
for path in (BASE_PATH, LINKS_PATH):
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} is missing. Mount Drive and run the prerequisite pilot cells."
        )

baseline = pd.read_csv(
    BASE_PATH,
    dtype={
        "imdbId": "string", "tmdbId": "string", "wikidata_id": "string",
        "plot": "string", "section": "string", "plot_source": "string",
    },
)
links_to_check = pd.read_csv(LINKS_PATH, dtype={"wikidata_id": "string"})
if baseline.empty or not baseline["movieId"].is_unique:
    raise ValueError("Expected a nonempty V3 pilot with unique MovieLens IDs.")

baseline_candidates = (
    baseline["has_candidate_text"].astype("string").str.lower().eq("true").fillna(False)
    & baseline["plot"].fillna("").str.strip().ne("")
)
missing_movies = baseline.loc[
    baseline["status"].eq("no_en_wikipedia") & ~baseline_candidates,
    ["movieId", "title", "wikidata_id"],
]
matched_links = links_to_check.drop(columns="title").merge(
    missing_movies, on=["movieId", "wikidata_id"], validate="many_to_one"
)
if len(matched_links) != len(links_to_check):
    raise ValueError("Sitelinks do not match V3. Refresh them with the preceding cell.")

priority = ["it", "fr", "es", "de", "pt", "nl", "ru", "pl", "ja", "zh"]
matched_links["priority"] = (
    matched_links["language"].map({lang: rank for rank, lang in enumerate(priority)})
    .fillna(len(priority))
)
matched_links = matched_links.sort_values(
    ["movieId", "priority", "language", "wikipedia_url"]
).drop_duplicates(["movieId", "wikipedia_url"])

multilingual_session = requests.Session()
multilingual_session.headers["User-Agent"] = (
    "MovieRecResearch/0.1 (https://github.com/Cxsee/AI-in-Industry-project)"
)
multilingual_session.mount(
    "https://",
    HTTPAdapter(max_retries=Retry(
        total=2, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504]
    )),
)
article_cache = {}


def parse_wikipedia(api_url, **params):
    """Read the MediaWiki parse result, allowing retries on replication lag."""
    for attempt in range(3):
        response = multilingual_session.get(
            api_url,
            params={"action": "parse", "format": "json", "maxlag": 5, **params},
            timeout=(10, 30),
            allow_redirects=False,
        )
        time.sleep(1)
        if response.is_redirect:
            raise RuntimeError("Unexpected redirect from the MediaWiki API endpoint.")
        response.raise_for_status()
        payload = response.json()
        error = payload.get("error")
        if error is None:
            return payload["parse"]
        if error.get("code") != "maxlag" or attempt == 2:
            raise RuntimeError(error.get("info", str(error)))
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(max(5, int(retry_after)) if retry_after.isdecimal() else 5)


def inspect_article(url, language):
    """Read headings and the revision ID once per article per run."""
    if url not in article_cache:
        article = urlparse(url)
        host = f"{language}.wikipedia.org"
        if (
            not re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", language)
            or article.scheme != "https"
            or article.netloc != host
            or not article.path.startswith("/wiki/")
            or article.path == "/wiki/"
        ):
            raise ValueError(f"Unexpected Wikipedia article URL: {url}")
        api_url = f"https://{host}/w/api.php"
        parsed = parse_wikipedia(
            api_url,
            page=unquote(article.path[len("/wiki/"):]).replace("_", " "),
            prop="tocdata|revid",
            redirects=1,
        )
        sections = [
            (section.get("index"), BeautifulSoup(section["line"], "html.parser").get_text(
                " ", strip=True
            ))
            for section in parsed["tocdata"]["sections"]
        ]
        article_cache[url] = (api_url, parsed["revid"], sections)
    return article_cache[url]


preferred_pages = matched_links.drop_duplicates("movieId")
preview_pages = preferred_pages.loc[preferred_pages["language"].ne("cy")].head(5)
inspection_rows = []
for page in preview_pages.itertuples(index=False):
    try:
        _, revision, sections = inspect_article(page.wikipedia_url, page.language)
        headings = " | ".join(heading for _, heading in sections)
        error = ""
    except (requests.RequestException, RuntimeError, KeyError, ValueError) as exc:
        revision, headings, error = None, "", str(exc)
    inspection_rows.append({
        "title": page.title, "language": page.language, "url": page.wikipedia_url,
        "revision": revision, "headings": headings, "error": error,
    })

if inspection_rows:
    display(pd.DataFrame(inspection_rows))
else:
    print("No preferred non-Welsh articles are available for the preview.")


In [ ]:
# 22. Extract explicit plot sections from available language editions
MIN_PLOT_CHARS = 150
COMMON_HEADINGS = {"plot", "plot summary", "synopsis", "storyline", "premise"}
LOCAL_HEADINGS = {
    "an": {"argumento", "trama"},
    "ca": {"argument", "sinopsi", "trama"},
    "cy": {"plot", "crynodeb", "crynodeb o'r plot"},
    "de": {"handlung", "filminhalt", "inhaltsangabe"},
    "es": {"argumento", "sinopsis", "trama", "resumen de la trama"},
    "et": {"sisu", "süžee"},
    "fi": {"juoni", "juonikuvaus"},
    "fr": {"synopsis", "synopsis détaillé", "résumé", "résumé détaillé", "intrigue"},
    "ht": {"rezime", "senopsis"},
    "it": {"trama", "sinossi", "intreccio"},
    "ja": {"あらすじ", "ストーリー", "物語"},
    "nl": {"verhaal", "plot", "verhaallijn", "inhoud", "synopsis"},
    "pl": {"fabuła", "opis fabuły"},
    "pt": {"enredo", "sinopse", "trama"},
    "ru": {"сюжет", "краткое содержание", "содержание фильма"},
    "sr": {"радња", "садржај", "radnja", "sadržaj"},
    "sv": {"handling", "handlingen", "intrig"},
    "ur": {"کہانی", "خلاصہ", "پلاٹ"},
    "zh": {"劇情", "剧情", "故事", "劇情簡介", "剧情简介"},
}


def normalize_heading(heading):
    heading = unicodedata.normalize("NFKC", heading).casefold().replace("’", "'")
    return " ".join(heading.split()).strip(" :：")


def clean_plot_html(html):
    soup = BeautifulSoup(html, "html.parser")
    for element in soup.select(
        "sup.reference, sup[typeof='mw:Extension/ref'], table, .mw-editsection, "
        ".thumb, figure, .hatnote, .navbox, .reference, .reflist, "
        ".noprint, .shortdescription, .ambox, style, script"
    ):
        element.decompose()
    prose = " ".join(p.get_text(" ", strip=True) for p in soup.find_all("p"))
    return re.sub(r"\s+([,.;:!?])", r"\1", " ".join(prose.split()))


def extract_multilingual_plot(url, language):
    api_url, revision, sections = inspect_article(url, language)
    revision_url = api_url.removesuffix("/api.php") + f"/index.php?oldid={revision}"
    accepted = COMMON_HEADINGS | LOCAL_HEADINGS.get(language, set())
    best = {
        "article_status": "no_plot_section",
        "plot_revision_url": revision_url,
        "section": "",
        "plot": "",
        "plot_chars": 0,
        "error": "",
    }
    for index, heading in sections:
        if not index or not index.isdecimal() or normalize_heading(heading) not in accepted:
            continue
        parsed = parse_wikipedia(api_url, oldid=revision, prop="text", section=index)
        plot = clean_plot_html(parsed["text"]["*"])
        if best["article_status"] == "no_plot_section" or len(plot) > best["plot_chars"]:
            best.update(
                section=heading, plot=plot, plot_chars=len(plot),
                article_status="short_plot" if plot else "empty_plot_section",
            )
        if len(plot) >= MIN_PLOT_CHARS:
            best["article_status"] = "recovered"
            break
    return best


audit_columns = [
    "movieId", "title", "language", "wikipedia_url", "plot_revision_url",
    "article_status", "section", "plot", "plot_chars", "error",
]
attempts = []
for movie_id, pages in matched_links.groupby("movieId", sort=False):
    for page in pages.itertuples(index=False):
        try:
            result = extract_multilingual_plot(page.wikipedia_url, page.language)
        except (requests.RequestException, RuntimeError, KeyError, ValueError) as exc:
            result = {
                "article_status": "article_error", "plot_revision_url": "",
                "section": "", "plot": "", "plot_chars": 0, "error": str(exc),
            }
        attempts.append({
            "movieId": movie_id, "title": page.title, "language": page.language,
            "wikipedia_url": page.wikipedia_url, **result,
        })
        print(f"{page.title} [{page.language}]: {result['article_status']}")
        if result["article_status"] == "recovered":
            break

article_audit = pd.DataFrame(attempts, columns=audit_columns)


In [ ]:
# 23. Consolidate the V4 pilot and report measured coverage
from tempfile import NamedTemporaryFile

AUDIT_PATH = PILOT_DIR / "multilingual_plot_audit.csv"
V4_PATH = PILOT_DIR / "plots_pilot_100_v4.csv"


def save_csv(frame, destination):
    """Replace a CSV only after the complete new file has been written."""
    with NamedTemporaryFile(dir=destination.parent, suffix=".tmp", delete=False) as temp:
        temp_path = Path(temp.name)
    try:
        frame.to_csv(temp_path, index=False)
        temp_path.replace(destination)
    finally:
        temp_path.unlink(missing_ok=True)


with_sitelinks = set(matched_links["movieId"])
recovered = article_audit.loc[article_audit["article_status"].eq("recovered")].copy()
recovered_ids = set(recovered["movieId"])
error_ids = set(article_audit.loc[
    article_audit["article_status"].eq("article_error"), "movieId"
])
unresolved_errors = error_ids - recovered_ids
without_sitelinks = set(missing_movies["movieId"]) - with_sitelinks
without_usable_section = with_sitelinks - recovered_ids - unresolved_errors

# Record films without sitelinks as well as all attempted articles.
no_link_rows = missing_movies.loc[
    missing_movies["movieId"].isin(without_sitelinks), ["movieId", "title"]
].assign(article_status="no_sitelink", plot_chars=0)
save_csv(pd.concat([article_audit, no_link_rows], ignore_index=True), AUDIT_PATH)
print(f"Audit saved to: {AUDIT_PATH}")

if not article_audit.empty and article_audit["article_status"].eq("article_error").all():
    raise RuntimeError(
        "Every article request failed. See the audit and rerun extraction before publishing V4."
    )

if not recovered["movieId"].is_unique:
    raise ValueError("Expected at most one recovered plot per MovieLens ID.")

pilot_v4 = baseline.copy().set_index("movieId", drop=False)
pilot_v4["has_candidate_text"] = baseline_candidates.to_numpy()
pilot_v4["plot_language"] = pd.Series(pd.NA, index=pilot_v4.index, dtype="string")
pilot_v4["plot_revision_url"] = pd.Series(pd.NA, index=pilot_v4.index, dtype="string")
pilot_v4["plot_wikipedia_url"] = pilot_v4["wikipedia_url"].astype("string").where(
    pilot_v4["has_candidate_text"]
)
pilot_v4.loc[pilot_v4["has_candidate_text"], "plot_language"] = "en"

if not recovered.empty:
    new_plots = recovered.set_index("movieId")
    for column in ["plot", "section", "plot_chars", "plot_revision_url"]:
        pilot_v4.loc[new_plots.index, column] = new_plots[column]
    pilot_v4.loc[new_plots.index, "plot_language"] = new_plots["language"]
    pilot_v4.loc[new_plots.index, "plot_wikipedia_url"] = new_plots["wikipedia_url"]
    pilot_v4.loc[new_plots.index, "plot_source"] = "wikipedia_multilingual_plot_section"
    pilot_v4.loc[new_plots.index, "status"] = "multilingual_plot_extracted"
    pilot_v4.loc[new_plots.index, "has_candidate_text"] = True

total = len(pilot_v4)
before = int(baseline_candidates.sum())
after = int(pilot_v4["has_candidate_text"].sum())
if after != before + len(recovered) or len(pilot_v4) != len(baseline):
    raise ValueError("V4 coverage or row count does not reconcile with V3.")
save_csv(pilot_v4, V4_PATH)

coverage = pd.DataFrame([
    ("V3", before, before / total),
    ("V4", after, after / total),
], columns=["pilot", "movies_with_candidate_text", "coverage"])
display(coverage.assign(coverage=coverage["coverage"].map("{:.1%}".format)))
print(f"Coverage gain: {100 * (after - before) / total:+.1f} percentage points")
print(f"Films investigated: {len(missing_movies)}")
print(f"Recovered: {len(recovered)}")
print(f"No alternative Wikipedia page: {len(without_sitelinks)}")
print(f"No usable plot section found: {len(without_usable_section)}")
print(f"Unresolved due to article errors: {len(unresolved_errors)}")
print(f"V4 saved to: {V4_PATH}")

if not recovered.empty:
    display(recovered[[
        "movieId", "title", "language", "section", "plot_chars", "plot_revision_url"
    ]])
